In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Setup and Vocabulary

In [36]:
SOS_token = 0
EOS_token = 1

training_pairs = [
    ("how do i reset my password ?", "you can reset it in settings"),
    ("hello", "hello"),
    ("hi", "hi how can i help you"),
    ("how are you", "i'm good"),
    ("goodbye", "goodbye see you"),
    ("thank you", "you're welcome"),
    ("where is the nearest store", "i can help with that"),
    ("i need help", "sure how can i help you")
]

# Dynamically build vocabularies from training data
def build_vocab(pairs, index):
    vocab = {"SOS": 0, "EOS": 1}
    for pair in pairs:
        for word in pair[index].lower().split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

input_vocab = build_vocab(training_pairs, 0)
output_vocab = build_vocab(training_pairs, 1)
index_to_word = {v: k for k, v in output_vocab.items()}

print(f"Input vocab size: {len(input_vocab)}")
print(f"Output vocab size: {len(output_vocab)}")

Input vocab size: 22
Output vocab size: 22


# 2. Encoder Model

In [22]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input_seq, hidden):
        embedded = self.embedding(input_seq).view(1, 1, -1)
        output, hidden = self.gru(embedded, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size)

# 3. Decoder Model

In [23]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_token, hidden):
        output = self.embedding(input_token).view(1, 1, -1)
        output = torch.relu(output)
        output, hidden = self.gru(output, hidden)
        prediction = self.softmax(self.out(output[0]))
        return prediction, hidden

# 4. Training Step Mechanics

In [24]:
def train_step(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    encoder_hidden = encoder.initHidden()

    # Encode
    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)

    # Decode
    decoder_hidden = encoder_hidden
    decoder_input = torch.tensor([[SOS_token]])
    loss = 0

    for di in range(target_length):
        decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
        loss += criterion(decoder_output, target_tensor[di])
        decoder_input = target_tensor[di] # Teacher forcing

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

# 5. Inference / Execution

In [25]:
def evaluate_chatbot(encoder, decoder, input_sentence):
    with torch.no_grad():
        # Convert input to tensor
        input_indices = [input_vocab[word.lower()] for word in input_sentence.split()]
        input_indices.append(EOS_token)
        input_tensor = torch.tensor(input_indices).view(-1, 1)

        # Encode
        encoder_hidden = encoder.initHidden()
        for ei in range(input_tensor.size(0)):
            _, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)

        context_vector = encoder_hidden

        # Decode
        decoder_input = torch.tensor([[SOS_token]])
        decoder_hidden = context_vector
        decoded_words = []

        for di in range(10): # Max length
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            top_value, top_index = decoder_output.data.topk(1)

            if top_index.item() == EOS_token:
                break
            else:
                decoded_words.append(index_to_word[top_index.item()])

            decoder_input = top_index.squeeze().detach()

        return " ".join(decoded_words)

# 6. Main Execution Block

In [26]:
if __name__ == "__main__":
    hidden_size = 256

    # Initialize models
    encoder = EncoderRNN(len(input_vocab), hidden_size)
    decoder = DecoderRNN(hidden_size, len(output_vocab))

    user_input = "how do i reset my password ?"

    # Evaluate the input before training
    response = evaluate_chatbot(encoder, decoder, user_input)

    print(f"User: {user_input}")
    print(f"Bot (Untrained Output): {response}")

User: how do i reset my password ?
Bot (Untrained Output): it it it it it it it it it it


# 7. Training Data

In [32]:
training_pairs = [
    ("how do i reset my password ?", "you can reset it in settings"),
    ("hello", "hello"),
    ("hi", "hi how can i help you"),
    ("how are you", "i'm good"),
    ("goodbye", "goodbye see you"),
    ("thank you", "you're welcome"),
    ("where is the nearest store", "i can help with that"),
    ("i need help", "sure how can i help you")
]

def tensor_from_sentence(vocab, sentence):
    indexes = [vocab[word.lower()] for word in sentence.split()]
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long).view(-1, 1)

# 8. Training Loop

In [33]:
def train_model(encoder, decoder, training_pairs, n_iters, print_every=100):
    encoder_optimizer = optim.SGD(encoder.parameters(), lr=0.01)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=0.01)
    criterion = nn.NLLLoss()

    print("Starting training...")
    for iter in range(1, n_iters + 1):
        total_loss = 0
        for training_pair in training_pairs:
            input_tensor = tensor_from_sentence(input_vocab, training_pair[0])
            target_tensor = tensor_from_sentence(output_vocab, training_pair[1])

            loss = train_step(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
            total_loss += loss

        average_loss = total_loss / len(training_pairs)

        if iter % print_every == 0:
            print(f"Iteration {iter}: Average Loss = {average_loss:.4f}")

    print("Training finished.")

# 9. Run Training and Evaluate

In [37]:
if __name__ == "__main__":
    hidden_size = 256

    # Initialize models with correct sizes from updated vocab
    encoder = EncoderRNN(len(input_vocab), hidden_size)
    decoder = DecoderRNN(hidden_size, len(output_vocab))

    # Run training
    n_iterations = 1000
    train_model(encoder, decoder, training_pairs, n_iterations, print_every=200)

    print("\n--- Evaluating Trained Chatbot ---")

    test_sentences = [
        "how do i reset my password ?",
        "hello",
        "where is the nearest store",
        "thank you"
    ]

    for sentence in test_sentences:
        response = evaluate_chatbot(encoder, decoder, sentence)
        print(f"User: {sentence}")
        print(f"Bot: {response}\n")

Starting training...
Iteration 200: Average Loss = 0.0092
Iteration 400: Average Loss = 0.0036
Iteration 600: Average Loss = 0.0022
Iteration 800: Average Loss = 0.0015
Iteration 1000: Average Loss = 0.0012
Training finished.

--- Evaluating Trained Chatbot ---
User: how do i reset my password ?
Bot: you can reset it in settings

User: hello
Bot: hello

User: where is the nearest store
Bot: i can help with that

User: thank you
Bot: you're welcome

